In [0]:
from datetime import datetime
from pyspark.sql.functions import col, date_format

# Optimize shuffle for small clusters
spark.conf.set("spark.sql.shuffle.partitions", 4)

# 1. Widgets & Setup
dbutils.widgets.text("customer_id", "")
dbutils.widgets.text("object_name", "")
dbutils.widgets.text("source_system", "salesforce")
dbutils.widgets.text("bucket_path", "")
dbutils.widgets.text("sf_catalog_table", "")

customer_id = dbutils.widgets.get("customer_id")
object_name = dbutils.widgets.get("object_name")
source_system = dbutils.widgets.get("source_system")
bucket_path = dbutils.widgets.get("bucket_path")
sf_catalog_table = dbutils.widgets.get("sf_catalog_table")

# Timestamp path
now = datetime.now()
yyyy, mm, dd, hh = now.strftime("%Y"), now.strftime("%m"), now.strftime("%d"), now.strftime("%H")
incremental_path = f"{bucket_path}landing-zone/{source_system}/{customer_id}/{object_name}/incremental/{yyyy}/{mm}/{dd}/{hh}"

print(f"🚀 Starting Incremental Load to: {incremental_path}")

# 2. Get the last watermark
try:
    watermark_df = spark.sql(f"""
    SELECT last_processed_timestamp
    FROM ingestion_metadata.watermarks
    WHERE source_system='{source_system}'
    AND object_name='{object_name}'
    """)
    rows = watermark_df.collect()
    if len(rows) == 0:
        raise Exception("Watermark missing.")
    watermark = rows[0][0]
except Exception as e:
    raise Exception(f"❌ Watermark not initialized. Please run historical load first. Error: {str(e)}")

print(f"📍 Last Watermark: {watermark}")

# 3. Pre-Fetch: Query Salesforce FIRST to get the NEW max timestamp
new_ts_df = spark.sql(f"""
SELECT MAX(SystemModstamp) as max_ts
FROM {sf_catalog_table}
WHERE SystemModstamp > TIMESTAMP('{watermark}')
""")

new_ts = new_ts_df.first()[0]

if new_ts is None:
    print("✅ No new records found in Salesforce. Exiting gracefully.")
    dbutils.notebook.exit("0")

print(f"🎯 New records found up to: {new_ts}")

# 4. Pull the data using BOTH bounds
df_incremental = spark.sql(f"""
SELECT *
FROM {sf_catalog_table}
WHERE SystemModstamp > TIMESTAMP('{watermark}')
  AND SystemModstamp <= TIMESTAMP('{new_ts}')
""")

# Format the Timestamp to exact String format 
df_formatted = df_incremental.withColumn("SystemModstamp", date_format(col("SystemModstamp"), "yyyy-MM-dd'T'HH:mm:ss.SSS'Z'"))

try:
    # 5. Write to S3
    (
        df_formatted
        .repartition(4)
        .write
        .mode("append")
        .format("parquet")
        .option("compression", "snappy")
        .save(incremental_path)
    )
    print("💾 Write to S3 completed successfully.")
    
except Exception as e:
    print(f"❌ Failed during write operation: {str(e)}")
    raise e

# 6. Update Watermark with 1-Minute Lookback
print("Updating watermark table safely using MERGE...")

# THE FIX: Use MERGE INTO to avoid concurrent overwrite errors
spark.sql(f"""
    MERGE INTO ingestion_metadata.watermarks AS target
    USING (
        SELECT '{source_system}' AS source_system, 
               '{object_name}' AS object_name, 
               TIMESTAMP('{new_ts}') - INTERVAL 1 MINUTE AS last_processed_timestamp
    ) AS source
    ON target.source_system = source.source_system AND target.object_name = source.object_name
    WHEN MATCHED THEN
        UPDATE SET target.last_processed_timestamp = source.last_processed_timestamp
    WHEN NOT MATCHED THEN
        INSERT (source_system, object_name, last_processed_timestamp)
        VALUES (source.source_system, source.object_name, source.last_processed_timestamp)
""")

print(f"✅ Watermark updated safely (minus 1 minute).")
dbutils.notebook.exit("Success")